<a href="https://colab.research.google.com/github/shishyan/GuruKulaDesam/blob/main/RemotePC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup Remote PC

In [68]:
import subprocess
import os
import time

# 1. Install code-server if it is not already installed
if not os.path.exists('/usr/bin/code-server') and not os.path.exists('/usr/local/bin/code-server'):
    print('[-] Installing code-server...')
    # Download and run the official installation script
    install_process = subprocess.run(
        'curl -fsSL https://code-server.dev/install.sh | sh',
        shell=True,
        capture_output=True,
        text=True
    )
    print(install_process.stdout)
    print('[-] code-server installation completed.')
else:
    print('[-] code-server is already installed.')

[-] code-server is already installed.


In [69]:
import subprocess
import time
import json
import os

PORT = "7860"

# 1. Kill any existing process on port 7860
print('[-] Stopping any process running on port 7860...')
subprocess.run(f'fuser -k {PORT}/tcp', shell=True, capture_output=True)
time.sleep(1)

# 2. Configure code-server User Settings to force Dark Mode as default theme
settings_dir = os.path.expanduser('~/.local/share/code-server/User')
os.makedirs(settings_dir, exist_ok=True)
settings_file = os.path.join(settings_dir, 'settings.json')

settings = {}
if os.path.exists(settings_file):
    try:
        with open(settings_file, 'r') as f:
            settings = json.load(f)
    except Exception:
        pass

# Inject default dark theme preferences
settings['workbench.colorTheme'] = 'Default Dark+'
with open(settings_file, 'w') as f:
    json.dump(settings, f, indent=2)
print('[+] Code-server settings updated to default Dark Mode.')

# 3. Launch code-server in the background
print(f'[-] Launching code-server on port {PORT}...')
code_server_process = subprocess.Popen([
    'code-server',
    '--bind-addr', f'0.0.0.0:{PORT}',
    '--auth', 'none',
    '--disable-telemetry'
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(3)
print('[-] code-server has been started successfully in Dark Mode.')

[-] Stopping any process running on port 7860...
[+] Code-server settings updated to default Dark Mode.
[-] Launching code-server on port 7860...
[-] code-server has been started successfully in Dark Mode.


### 📓 Setup Sequence Complete
All key server configurations are consolidated below.

In [70]:
from google.colab import drive
import os

# 1. Mount Google Drive
print('[-] Mounting Google Drive...')
drive.mount('/content/drive')

# 2. Establish the target directory path
target_dir = '/content/drive/MyDrive/GuruKulaDesam'
os.makedirs(target_dir, exist_ok=True)
file_path = os.path.join(target_dir, 'Coder_Server_Setup_Guide.md')

# 3. Create the comprehensive markdown documentation including CLI installation, symlinks, and updated parameters
doc_content = """# Coder Server Setup & Cloudflare Tunnel Guide

This guide explains how to install, configure, and secure a web-accessible instance of **code-server** (VS Code in the browser) on port `7860` routed securely using a custom domain via a locally-managed **Cloudflare Tunnel**.

---

## 1. Prerequisites
* A Linux or Colab environment
* A Cloudflare account with a configured Tunnel and a custom domain (e.g., `code.shishyan.in`)
* Your Cloudflare Zero Trust Tunnel Token

---

## 2. Install antigravity-cli First
Before setting up any environment parameters, the main `agy` CLI/daemon framework must be installed via `antigravity-cli`:
```bash
pip install antigravity-cli --quiet
```

### Python Automation Script:
```python
import subprocess

print('[-] Installing antigravity-cli...')
subprocess.run('pip install antigravity-cli --quiet', shell=True)
```

---

## 3. Create Symlink for agyd
Since `antigravity-cli` installs the executable as `agy`, create a symbolic link so any script executing `agyd` is routed correctly:
```bash
ln -sf /usr/local/bin/agy /usr/local/bin/agyd
```

### Python Automation Script:
```python
import os
import subprocess

if os.path.exists('/usr/local/bin/agy') and not os.path.exists('/usr/local/bin/agyd'):
    print('[-] Creating symlink from agyd to agy...')
    subprocess.run('ln -sf /usr/local/bin/agy /usr/local/bin/agyd', shell=True)
```

---

## 4. Installation of code-server
Install `code-server` directly onto the host machine via the official installer:
```bash
curl -fsSL https://code-server.dev/install.sh | sh
```

### Python Automation Script:
```python
import subprocess
import os

if not os.path.exists('/usr/bin/code-server') and not os.path.exists('/usr/local/bin/code-server'):
    print('[-] Installing code-server...')
    install_process = subprocess.run(
        'curl -fsSL https://code-server.dev/install.sh | sh',
        shell=True,
        capture_output=True,
        text=True
    )
    print(install_process.stdout)
    print('[-] code-server installation completed.')
else:
    print('[-] code-server is already installed.')
```

---

## 5. Running code-server
To run `code-server` as a background process listening on port `7860` with password authentication bypassed (use with caution or configure a password inside your configurations):
```bash
code-server --bind-addr 0.0.0.0:7860 --auth none --disable-telemetry
```

### Python Automation Script:
```python
import subprocess
import time

PORT = "7860"

# 1. Kill any existing process listening on port 7860 to avoid bind conflicts
print('[-] Stopping any process running on port 7860...')
subprocess.run(f'fuser -k {PORT}/tcp', shell=True, capture_output=True)
time.sleep(1)

# 2. Launch code-server in the background
print(f'[-] Launching code-server on port {PORT}...')
code_server_process = subprocess.Popen([
    'code-server',
    '--bind-addr', f'0.0.0.0:{PORT}',
    '--auth', 'none',
    '--disable-telemetry'
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(3)
print('[-] code-server has been started.')
```

---

## 6. Install Cloudflare Daemon (cloudflared)
Download and authorize execution permissions for the latest official Cloudflare Tunnel daemon:
```bash
wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
chmod +x /content/cloudflared
```

### Python Automation Script:
```python
import os
import urllib.request

if not os.path.exists('/content/cloudflared'):
    print('Downloading cloudflared...')
    urllib.request.urlretrieve(\'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\', \'/content/cloudflared\')
    os.chmod(\'/content/cloudflared\', 0o755)

print(\'cloudflared is ready.\')
```

---

## 7. Configure Shell Aliases (agy and agyd)
To automate the environment setup and bypass prompt loops, configure the shell alias mapping for both terms:

```python
import os
bash_profile = os.path.expanduser(\'~/.bashrc\')
with open(bash_profile, \'r\') as f:
    lines = f.readlines()

clean_lines = [line for line in lines if \'agy\' not in line]

with open(bash_profile, \'w\') as f:
    f.writelines(clean_lines)
    f.write("\\n# Custom Setup Aliases\\nalias agyd=\\'agy --dangerously-skip-permissions\\\'\\nalias agy=\\'agy --dangerously-skip-permissions\\\'\\n")
print(\'[+] Shell aliases configured.\')
```

---

## 8. Starting the Locally-Managed Tunnel
To bypass dashboard-managed limitations and route all incoming traffic directly to your `code-server` process on port `7860` via your tunnel token:
```bash
/content/cloudflared tunnel --no-autoupdate --url http://localhost:7860 run --token <YOUR_CLOUDFLARE_TUNNEL_TOKEN>
```
"""

# 4. Write document to target Google Drive directory
with open(file_path, 'w') as f:
    f.write(doc_content)

print(f'\n✨ Documentation Guide successfully updated with symlink and correct antigravity-cli installation: {file_path}')

[-] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✨ Documentation Guide successfully updated with symlink and correct antigravity-cli installation: /content/drive/MyDrive/GuruKulaDesam/Coder_Server_Setup_Guide.md


In [71]:
# @title
import json
import os

# Define the cells for our new Jupyter Notebook (.ipynb)
notebook_data = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# Coder Server Setup & Cloudflare Tunnel Automation\n",
                "This notebook installs `code-server`, installs the `antigravity-cli` framework, maps the correct symlinks, and establishes a secure Cloudflare Tunnel connection routing to port `7860` under a custom domain."
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 1: Install antigravity-cli Tool"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import subprocess\n",
                "print('[-] Installing antigravity-cli tool...')\n",
                "subprocess.run('pip install antigravity-cli --quiet', shell=True)\n",
                "print('[+] antigravity-cli installation complete.')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 2: Create Symlink for agyd\n",
                "Create a symbolic link mapping `agyd` to `agy` so that any scripts calling `agyd` are routed to the proper executable."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import os\n",
                "import subprocess\n",
                "if os.path.exists('/usr/local/bin/agy') and not os.path.exists('/usr/local/bin/agyd'):\n",
                "    print('[-] Creating symlink from agyd to agy...')\n",
                "    subprocess.run('ln -sf /usr/local/bin/agy /usr/local/bin/agyd', shell=True)\n",
                "print('[+] Symlink check complete.')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 3: Install code-server"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import subprocess\n",
                "import os\n",
                "\n",
                "if not os.path.exists('/usr/bin/code-server') and not os.path.exists('/usr/local/bin/code-server'):\n",
                "    print('[-] Installing code-server...')\n",
                "    install_process = subprocess.run(\n",
                "        'curl -fsSL https://code-server.dev/install.sh | sh',\n",
                "        shell=True,\n",
                "        capture_output=True,\n",
                "        text=True\n",
                "    )\n",
                "    print(install_process.stdout)\n",
                "    print('[-] code-server installation completed.')\n",
                "else:\n",
                "    print('[-] code-server is already installed.')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 4: Start code-server in the Background"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import subprocess\n",
                "import time\n",
                "\n",
                "PORT = '7860'\n",
                "\n",
                "# Kill conflicting processes on port 7860\n",
                "print('[-] Stopping any process running on port 7860...')\n",
                "subprocess.run(f'fuser -k {PORT}/tcp', shell=True, capture_output=True)\n",
                "time.sleep(1)\n",
                "\n",
                "# Launch code-server in background with password authentication off\n",
                "print(f'[-] Launching code-server on port {PORT}...')\n",
                "code_server_process = subprocess.Popen([\n",
                "    'code-server',\n",
                "    '--bind-addr', f'0.0.0.0:{PORT}',\n",
                "    '--auth', 'none',\n",
                "    '--disable-telemetry'\n",
                "], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n",
                "\n",
                "time.sleep(3)\n",
                "print('[-] code-server has been started successfully.')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 5: Download Cloudflare Tunnel Daemon (cloudflared)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import os\n",
                "import urllib.request\n",
                "\n",
                "if not os.path.exists('/content/cloudflared'):\n",
                "    print('[-] Downloading cloudflared...')\n",
                "    urllib.request.urlretrieve(\'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\', \'/content/cloudflared\')\n",
                "    os.chmod('/content/cloudflared', 0o755)\n",
                "\n",
                "print('[-] cloudflared is ready.')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 6: Configure Terminal Environment (Bypass Permission Prompts)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import os\n",
                "bash_profile = os.path.expanduser('~/.bashrc')\n",
                "with open(bash_profile, 'r') as f:\n",
                "    lines = f.readlines()\n",
                "clean_lines = [line for line in lines if 'agy' not in line]\n",
                "with open(bash_profile, 'w') as f:\n",
                "    f.writelines(clean_lines)\n",
                "    f.write(\"\\n# Custom Setup Aliases\\nalias agyd=\\'agy --dangerously-skip-permissions\\\'\\nalias agy=\\'agy --dangerously-skip-permissions\\\'\\n\")\n",
                "print('[+] Terminal aliases mapping configured successfully.')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Step 7: Run Locally-Managed Cloudflare Tunnel\n",
                "Paste your Cloudflare Zero Trust Tunnel Token below to route traffic from your custom domain to port `7860`."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import subprocess\n",
                "import os\n",
                "import time\n",
                "\n",
                "CLOUDFLARE_TUNNEL_TOKEN = 'eyJhIjoiNjc2Y2NlNGFjM2FkMmJkNTIzNGYzNGU5YzhlMzI4NTEiLCJ0IjoiNzZiMGVlY2EtOWYxOC00YzFkLTk0NTYtMTQzZDljNTFmMjZmIiwicyI6IkVyOEYrS1NQa0ZFZkZMOUIzVEUwa2tMd2NhWFcvR1pWa1NXTWlBTFU5czg9In0='\n",
                "\n",
                "print('[-] Killing any existing cloudflared processes...')\n",
                "subprocess.run(['pkill', '-f', 'cloudflared'])\n",
                "time.sleep(2)\n",
                "\n",
                "print('[-] Starting cloudflared tunnel pointing directly to localhost:7860...')\n",
                "log_file = open('token_tunnel.log', 'w')\n",
                "\n",
                "token_tunnel_process = subprocess.Popen(\n",
                "    [\n",
                "        '/content/cloudflared',\n",
                "        'tunnel',\n",
                "        '--no-autoupdate',\n",
                "        '--url', 'http://localhost:7860',\n",
                "        'run',\n",
                "        '--token', CLOUDFLARE_TUNNEL_TOKEN\n",
                "    ],\n",
                "    stdout=log_file,\n",
                "    stderr=log_file\n",
                ")\n",
                "\n",
                "time.sleep(6)\n",
                "print('[-] Tunnel restarted. Checking latest logs:')\n",
                "if os.path.exists('token_tunnel.log'):\n",
                "    with open('token_tunnel.log', 'r') as f:\n",
                "        print(''.join(f.readlines()[-15:]))\n",
                "else:\n",
                "    print('[!] Log file not found.')"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3 (ipykernel)",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 2
}

# Save the generated .ipynb notebook into Google Drive
target_notebook_path = '/content/drive/MyDrive/GuruKulaDesam/VMBuild.ipynb'
with open(target_notebook_path, 'w') as f:
    json.dump(notebook_data, f, indent=2)

print(f'[+] Clean Jupyter Notebook (with correct antigravity-cli & symlink setup) built and saved to: {target_notebook_path}')

[+] Clean Jupyter Notebook (with correct antigravity-cli & symlink setup) built and saved to: /content/drive/MyDrive/GuruKulaDesam/VMBuild.ipynb


In [76]:
import subprocess
import os
import time

# Securely hardcoded token to persist across runtime reconnections
CLOUDFLARE_TUNNEL_TOKEN = 'eyJhIjoiNjc2Y2NlNGFjM2FkMmJkNTIzNGYzNGU5YzhlMzI4NTEiLCJ0IjoiNzZiMGVlY2EtOWYxOC00YzFkLTk0NTYtMTQzZDljNTFmMjZmIiwicyI6IkVyOEYrS1NQa0ZFZkZMOUIzVEUwa2tMd2NhWFcvR1pWa1NXTWlBTFU5czg9In0='

# Restart the tunnel using the token but forcing the service routing via command line arguments
print('[-] Killing any existing cloudflared processes...')
subprocess.run(['pkill', '-f', 'cloudflared'])
time.sleep(2)

print('[-] Starting cloudflared tunnel pointing directly to localhost:7860...')
log_file = open('token_tunnel.log', 'w')

# By running with your token and specifying --url, we route your managed domain directly to port 7860
token_tunnel_process = subprocess.Popen(
    [
        '/content/cloudflared',
        'tunnel',
        '--no-autoupdate',
        '--url', 'http://localhost:7860',
        'run',
        '--token', CLOUDFLARE_TUNNEL_TOKEN
    ],
    stdout=log_file,
    stderr=log_file
)

time.sleep(6)
print('[-] Tunnel restarted. Checking latest logs:')
if os.path.exists('token_tunnel.log'):
    with open('token_tunnel.log', 'r') as f:
        print(''.join(f.readlines()[-15:]))
else:
    print('[!] Log file not found.')

[-] Killing any existing cloudflared processes...
[-] Starting cloudflared tunnel pointing directly to localhost:7860...
[-] Tunnel restarted. Checking latest logs:
2026-10-21T14:33:33Z INF Initial protocol quic
2026-10-21T14:33:33Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-10-21T14:33:33Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026/10/21 14:33:33 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
2026-10-21T14:33:33Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-10-21T14:33:33Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-10-21T14:33:33Z INF Starting metrics server on 127.0.0.1:20242/metrics
2026-10-21T14:33:33Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveID(65074) CurveP256] connIndex=0 event=0 ip=198.41.200.53
2026-10-21T14:33:33Z INF Registered tunnel connection connIndex=0 c

In [74]:
import subprocess

print('[-] Installing antigravity-cli...')
# Install the correct antigravity-cli tool
try:
    install_result = subprocess.run('pip install antigravity-cli --quiet', shell=True, capture_output=True, text=True)
    print(install_result.stdout)
    print('[+] antigravity-cli installation complete.')
except Exception as e:
    print(f'[!] Failed to install antigravity-cli: {e}')

[-] Installing antigravity-cli...

[+] antigravity-cli installation complete.


In [75]:
import os
import subprocess

# 1. Symlink agyd to agy so that any script calling agyd executes agy instead
if os.path.exists('/usr/local/bin/agy') and not os.path.exists('/usr/local/bin/agyd'):
    print('[-] Creating symlink from agyd to agy...')
    subprocess.run('ln -sf /usr/local/bin/agy /usr/local/bin/agyd', shell=True)

# 2. Re-write the ~/.bashrc aliases correctly so both commands bypass permissions
bash_profile = os.path.expanduser('~/.bashrc')
with open(bash_profile, 'r') as f:
    lines = f.readlines()

# Filter out old custom setup aliases to prevent duplication
clean_lines = [line for line in lines if 'agy' not in line]

with open(bash_profile, 'w') as f:
    f.writelines(clean_lines)
    f.write("\n# Custom Setup Aliases\nalias agyd='agy --dangerously-skip-permissions'\nalias agy='agy --dangerously-skip-permissions'\n")

print('[+] Successfully created symlink and configured ~/.bashrc aliases for both agy and agyd.')

[+] Successfully created symlink and configured ~/.bashrc aliases for both agy and agyd.
